# Results Explorer

Update `results_paths` below to point to the result JSONLs you want to inspect.

In [ ]:
import json, random, re, pathlib
from typing import List, Dict, Any
from metrics import analyze_pointwise_results

def load_records(path: pathlib.Path) -> List[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def summarize_file(path: pathlib.Path) -> Dict[str, Any]:
    recs = load_records(path)
    meta = next((r.get('metadata') or {} for r in recs if (r.get('metadata') or {})), {})
    scores = []
    for r in recs:
        s = (r.get('result') or {}).get('score')
        if isinstance(s, (int, float)):
            scores.append(s)
    coverage = len(scores)/len(recs) if recs else 0.0
    metrics = analyze_pointwise_results(path)
    summary = metrics.get('summary', {})
    dist = metrics.get('distance_from_original', {})
    m = re.search(r'-(p\d+|temp[0-9p]+)', path.stem)
    sweep_key = m.group(1) if m else None
    return {
        'path': str(path),
        'model_name': meta.get('model_name') or meta.get('model') or path.stem,
        'sweep_key': sweep_key,
        'examples': summary.get('examples_in_file'),
        'score_min': summary.get('score_min'),
        'score_max': summary.get('score_max'),
        'coverage': round(coverage,4),
        'mean_MAD_O': dist.get('mean_MAD_O'),
        'mean_RMSD_O': dist.get('mean_RMSD_O'),
    }

def display_table(rows: List[Dict[str, Any]]):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        from pprint import pprint
        for row in rows:
            pprint(row); print()

def plot_sweep(rows: List[Dict[str, Any]]):
    try:
        import matplotlib.pyplot as plt
    except Exception as e:
        print('matplotlib not installed:', e); return
    rows = [r for r in rows if r.get('sweep_key')]
    if not rows:
        print('No sweep keys to plot.'); return
    x=[r['sweep_key'] for r in rows]
    mad=[r.get('mean_MAD_O') for r in rows]
    rmsd=[r.get('mean_RMSD_O') for r in rows]
    plt.figure(figsize=(8,4))
    plt.plot(x, mad, marker='o', label='mean_MAD_O')
    plt.plot(x, rmsd, marker='s', label='mean_RMSD_O')
    plt.xticks(rotation=45); plt.grid(True); plt.tight_layout(); plt.legend();

def show_random_record(path: pathlib.Path, seed: int=None, idx: int=None):
    recs = load_records(path)
    if not recs:
        print(f'No records in {path}'); return
    rng = random.Random(seed)
    chosen_idx = idx if idx is not None else rng.randrange(len(recs))
    rec = recs[chosen_idx]
    print(f'Path: {path}\nIndex: {chosen_idx}/{len(recs)}\nID: {rec.get('id') or rec.get('data',{}).get('id')}')
    print('
Prompt messages:')
    for msg in rec.get('prompt', []):
        if isinstance(msg, dict):
            role = msg.get('role', '?'); content = msg.get('content','')
            print(f'[{role}] {content[:400]}')
    print('
Judge response:'); print(rec.get('response'))
    print('
Parsed result:'); print(rec.get('result'))


In [ ]:
# Set the result files to analyze. Add your BigGen tok120 L3200 JSONLs once copied locally.
results_paths = [
    pathlib.Path('results/flask/pointwise_results.jsonl'),
    pathlib.Path('results/flask/pointwise_results.perturbed.jsonl'),
]
results_paths = [p for p in results_paths if p.exists()]
results_paths

In [ ]:
rows = [summarize_file(p) for p in results_paths]
display_table(rows)
plot_sweep(rows)

In [ ]:
if results_paths:
    show_random_record(results_paths[0], seed=42)
else:
    print('No results_paths found; add files above.')